# 12 · Mapa "mente": área, eleitorado e votos dos municípios vencidos

**Três medidas, cada uma com denominador próprio** (sem exterior):
- **% da área**: área dos municípios vencidos ÷ área total da malha BR. CRS **Albers equal-area** (`+proj=aea +lat_1=-2 +lat_2=-22 +lat_0=-12 +lon_0=-54`, GRS80). A EPSG:5880 do briefing é Polyconic, não equivalente, então não foi usada (D-022).
- **% do eleitorado**: eleitorado dos vencidos ÷ eleitorado BR.
- **% dos votos**: votos **do próprio candidato** nos municípios que venceu ÷ votos nacionais **dele**.

**Ressalva**: área é proxy geográfico. Município grande e pouco povoado pesa muito na área e quase nada no eleitorado. Os dois empates exatos (SP 62448, TO 73555) não têm vencedor e ficam fora das contagens.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
b = cp.base_2026()
res = cp.resumo_mapa_mente(b)
print(
    res[
        [
            "nr_candidato",
            "n_municipios_vencidos",
            "pct_municipios_vencidos",
            "pct_area_sobre_area_br",
            "pct_eleitorado_sobre_eleitorado_br",
            "pct_votos_do_candidato_nos_vencidos",
        ]
    ]
    .round(2)
    .to_string(index=False)
)

In [ ]:
r = res[res.nr_candidato.isin([13, 22])].copy()
r["candidato"] = r.nr_candidato.map({13: "PT (Lula)", 22: "PL (Flávio)"})
medidas = {
    "pct_municipios_vencidos": "% dos municípios",
    "pct_area_sobre_area_br": "% da área",
    "pct_eleitorado_sobre_eleitorado_br": "% do eleitorado",
    "pct_votos_do_candidato_nos_vencidos": "% dos votos do candidato",
}
fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(medidas))
w = 0.38
for i, (_, row) in enumerate(r.iterrows()):
    vals = [row[k] for k in medidas]
    ax.bar(
        x + (i - 0.5) * w,
        vals,
        w,
        color=g.cor_candidato(int(row.nr_candidato)),
        label=row.candidato,
    )
    for xi, v in zip(x + (i - 0.5) * w, vals):
        ax.text(xi, v + 0.8, f"{v:.1f}", ha="center", fontsize=8)
ax.set_xticks(x, list(medidas.values()))
ax.set_ylabel("%")
ax.legend(frameon=False)
ax.set_title("Municípios vencidos: % de municípios, área, eleitorado e votos", loc="left")
ax.spines[["top", "right"]].set_visible(False)
g.salvar(fig, "mapa_mente_area_eleitorado_votos")
plt.show()

In [ ]:
mv = cp.malha_vencedores(b)
mv["vencedor"] = mv["nr_vencedor"].map(
    lambda v: "PL (22)" if v == 22 else ("PT (13)" if v == 13 else "empate (sem vencedor)")
)
cores = {
    "PT (13)": g.cor_candidato(13),
    "PL (22)": g.cor_candidato(22),
    "empate (sem vencedor)": "#ffffff",
}
fig = g.mapa_categorico(
    mv, "vencedor", cores, "1º turno 2026: quem venceu em cada município (sem exterior)"
)
g.salvar(fig, "mapa_mente_vencedores")
plt.show()